In [1]:
from pathlib import Path
import sys
import tomllib

from math import sqrt
from statsmodels.stats.power import TTestPower

import pandas as pd

In [2]:
# Navigiert vom Notebook-Ordner eine Ebene nach oben zum Projekt-Root
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

print(root_dir)

/Users/msyl16/Desktop/ai_semgrep_sicherheitsluecken


In [3]:
# Pfad zur config.toml definieren (aus dem Notebook-Ordner eine Ebene höher)
config_path = Path.cwd().parent / "config.toml"

with open(config_path, "rb") as f:
    config = tomllib.load(f)

# config ist jetzt ein normales Python-Dictionary
print(config)

{'semgrep_deeper': {'command': ['semgrep', 'scan', '--config=p/owasp-top-ten', '--config=p/default', '--json']}, 'llm': {'key': 'sk-D0F62rXlHHkGZlinlQPItA', 'model': 'jlu/qwen3-coder-next'}, 'semgrep': {'command': ['semgrep', 'scan', '--config=auto', '--json']}, 'remove_files': {'discard_extensions': ['.png', '.jpg', '.jpeg', '.gif', '.svg', '.ico', '.webp', '.avif', '.mp3', '.mp4', '.webm', '.ogg', '.ogv', '.swf', '.ai', '.woff', '.woff2', '.ttf', '.eot', '.otf', '.scss', '.sass', '.css', '.map', '.pdf', '.epub', '.mobi', '.doc', '.odt', '.pptx', '.xlsx', '.ods', '.tex', '.rst', '.txt', '.puml', '.drawio', '.excalidraw', '.csv', '.zip', '.jar', '.war', '.exe', '.pyc', '.snap', '.mbz', '.imscc', '.ds_store', '.keep', '.gitkeep', '.nojekyll', '.lock', '.prettierrc', '.prettierignore', '.nvmrc', '.editorconfig', '.gitattributes', '.codespellignore', '.babelrc', '.browserslistrc', '.rspec', '.ruby-version', '.rbenv-gemsets', '.csproj', '.sln', '.mf', '.xsd', '.wsdl', '.src', '.tf-disable'

In [4]:
from src import data_preprocessing

# 1. Import Daten

In [5]:
df = pd.read_parquet("../data/parquet/df_filtered.parquet")

In [6]:
df.shape

(76622, 16)

# 2. Stichprobengröße ermitteln

Power Analysis: https://www.geeksforgeeks.org/python/introduction-to-power-analysis-in-python/

In [7]:
alpha = 0.05
power = 0.8
effect_size=0.2

In [8]:
obj = TTestPower()

In [9]:
sample_size = obj.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    alternative="two-sided"
)

In [10]:
sample_size

198.150820955562

# 3. Zufällige Stichprobe aus Source Code

* Das Sampling wird auf Basis der `item_type` durchgeführt
* Einbindung nur von **Source Code** - `primary_code`
* Datei sollte mindestens 1 Zeile an Code haben

In [11]:
df = df[df.item_type == "primary_code"]

df = df[df.lines_of_code > 1.]

In [12]:
df_sample = (
    df
    .sample(
        n=200, # Statt 199 -> direkt 200
        random_state=7
    )
)

In [13]:
df_sample

,source_markdown,name_file,id_file,programming_language,file_size_byte,item_type,lines_of_code,logical_lines_of_code,id_name,title_name,author,institution,medium,level,year,document_id
38653,index.md,index_block_1.js,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,JavaScript,300,primary_code,18,10,Learn Web Development_mdn_Online-course_2026,Learn Web Development,None,mdn,Online-course,beginner,2026.0,38653_Learn Web Development_index_block_1.js
81358,README.md,README_block_12.js,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,JavaScript,500,primary_code,7,4,Microsoft Web Development for Beginner_Microso...,Microsoft Web Development for Beginner,None,Microsoft,Online-course,beginner,NaN,81358_Microsoft Web Development for Beginner_R...
78442,README.md,README_block_6.sh,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,Shell,49,primary_code,2,1,Microsoft Web Development for Beginner_Microso...,Microsoft Web Development for Beginner,None,Microsoft,Online-course,beginner,NaN,78442_Microsoft Web Development for Beginner_R...
26690,index.md,index_block_6.js,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,JavaScript,320,primary_code,15,11,Learn Web Development_mdn_Online-course_2026,Learn Web Development,None,mdn,Online-course,beginner,2026.0,26690_Learn Web Development_index_block_6.js
38443,index.md,index_block_1.js,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,JavaScript,130,primary_code,5,4,Learn Web Development_mdn_Online-course_2026,Learn Web Development,None,mdn,Online-course,beginner,2026.0,38443_Learn Web Development_index_block_1.js
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92880,README.md,README_block_9.js,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,JavaScript,482,primary_code,6,3,Microsoft Web Development for Beginner_Microso...,Microsoft Web Development for Beginner,None,Microsoft,Online-course,beginner,NaN,92880_Microsoft Web Development for Beginner_R...
76901,README.md,README_block_13.js,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,JavaScript,17482,primary_code,340,271,Microsoft Web Development for Beginner_Microso...,Microsoft Web Development for Beginner,None,Microsoft,Online-course,beginner,NaN,76901_Microsoft Web Development for Beginner_R...
41631,index.md,index_block_1.sh,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,Shell,673,primary_code,14,10,Learn Web Development_mdn_Online-course_2026,Learn Web Development,None,mdn,Online-course,beginner,2026.0,41631_Learn Web Development_index_block_1.sh
58470,None,ToggleButton.js,/Users/msyl16/Desktop/ai_semgrep_sicherheitslu...,JavaScript,270,primary_code,12,10,The Fullstack Developer Path_Scrimba_Online-co...,The Fullstack Developer Path,None,Scrimba,Online-course,beginner,NaN,58470_The Fullstack Developer Path_ToggleButto...


# 4. Stichprobe in einen neuen Ordner `SampleData` einfügen

In [14]:
data_preprocessing.copy_and_rename_file(
    destination_dir="../data/SampleData/",
    df=df_sample,
    path_col="id_file",
    new_name="document_id"
)

Cache result

In [15]:
df_sample.to_parquet("../data/parquet/df_sample.parquet")